# 00 — Configurações Compartilhadas

Execute **antes de qualquer outro notebook**.

### Segurança de credenciais
As credenciais AWS são lidas via **Databricks Secrets** — nunca hardcoded.

**Como configurar (uma única vez via Databricks CLI):**
```bash
databricks secrets create-scope --scope ifood-aws
databricks secrets put --scope ifood-aws --key access-key
databricks secrets put --scope ifood-aws --key secret-key
databricks secrets put --scope ifood-aws --key bucket
```

## Célula 1 — Imports

In [8]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType
print("Imports OK")

ModuleNotFoundError: No module named 'pyspark'

## Célula 2 — Credenciais via Databricks Secrets + Paths

In [5]:
# Databricks Secrets: credenciais nunca aparecem no codigo
# Configurar via CLI antes de rodar (ver instrucoes no topo)
S3_BUCKET = dbutils.secrets.get(scope="ifood-aws", key="bucket")

# Paths S3 — acesso via IAM Role (ifood-s3-credential + ifood-s3-location)
BRONZE_S3 = f"s3://{S3_BUCKET}/bronze/nyc_taxi/yellow/"
SILVER_S3 = f"s3://{S3_BUCKET}/silver/nyc_taxi/yellow/"

# Unity Catalog
CATALOG      = "ifood_catalog"
BRONZE_TABLE = f"{CATALOG}.bronze.yellow_taxi"
SILVER_TABLE = f"{CATALOG}.silver.yellow_taxi"

# NYC TLC
TLC_BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
YEAR         = "2023"
MONTHS       = ["01", "02", "03", "04", "05"]

spark.sql(f'USE CATALOG {CATALOG}')

print(f"Catalog : {CATALOG}")
print(f"Bucket  : s3://{S3_BUCKET}")
print(f"Bronze  : {BRONZE_TABLE}")
print(f"Silver  : {SILVER_TABLE}")
print(f"Spark   : {spark.version}")
print("Config OK!")

NameError: name 'dbutils' is not defined

## Célula 3 — Validar acesso S3 via External Location

In [ ]:
print("Verificando External Location...")
spark.sql("SHOW EXTERNAL LOCATIONS").filter(
    "url LIKE '%ifood-case-datalake%'"
).show(truncate=False)
try:
    files = dbutils.fs.ls(f"s3://{S3_BUCKET}/")
    print(f"Acesso S3 OK — {len(files)} pasta(s) encontrada(s)")
    for f in files:
        print(f"  {f.path}")
except Exception as e:
    print(f"Erro: {e}")